### DSCI 410L Project data cleaning

# 1
Load the data files

In [17]:
import pandas as pd
import glob
import datetime

# MCSLC

In [61]:
#read file
mcslc = pd.read_csv("MCSLC.csv")
#subsection the DF to only necessary columns
mcslc_clean = mcslc[["ID", "City", "Dispatch Date & Time", "Disposition"]].dropna()

#Clean closed_as column
mcslc_clean = mcslc_clean.rename(columns={"Disposition": "closed_as"}) #renaming the disposition column for consistancy between dataframes
mcslc_clean["closed_as"] = mcslc_clean["closed_as"].str.upper()
#change cmergency department labal to HOSPITAL
mcslc_clean.replace('EMERGENCY DEPARTMENT', 'HOSPITAL', inplace=True)


#Clean City Column
mcslc_clean.drop(mcslc_clean[mcslc_clean["City"].isin(["Unknown", "Other", "Out of County"])].index, inplace=True)

#Change the datetime column to actual datetime variables
mcslc_clean["calltime"]= pd.to_datetime(mcslc_clean["Dispatch Date & Time"], format="%m/%d/%y %H:%M")

mcslc_eug = mcslc_clean.where(mcslc_clean["City"] == "Eugene").dropna()
mcslc_spd = mcslc_clean.where(mcslc_clean["City"] == "Springfield").dropna()

mcslc_spd.head()

,ID,City,Dispatch Date & Time,closed_as,calltime
33,14472.0,Springfield,11/4/25 5:42,HOSPITAL,2025-11-04 05:42:00
39,12245.0,Springfield,8/22/24 0:00,HOSPITAL,2024-08-22 00:00:00
48,10052.0,Springfield,1/14/25 15:04,HOSPITAL,2025-01-14 15:04:00
49,10062.0,Springfield,1/16/25 15:19,HOSPITAL,2025-01-16 15:19:00
51,10070.0,Springfield,2/1/25 16:31,HOSPITAL,2025-02-01 16:31:00


# Eugene


In [ ]:
url = ...
files = glob.glob(url)
eug = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
eug.head(3)

C:\Users\brook\AppData\Local\Temp\ipykernel_18448\1743683478.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  eug = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
C:\Users\brook\AppData\Local\Temp\ipykernel_18448\1743683478.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  eug = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)


,yr,agency,service,inci_id,calltime,case_id,callsource,nature,closecode,closed_as,secs_to_disp,secs_to_arrv,secs_to_close,disp,arrv,priority,primeunit,units_dispd,units_arrived,month
0,2015,EPD,LAW,15000001,2015-01-01 00:00:00.000,NaN,SELF,PERSON STOP,ASST,ASSISTED,0.0,0.0,217,1,1,6,_5E48,1,1,NaN
1,2015,EPD,LAW,15000002,2015-01-01 00:00:44.000,NaN,SELF,FIGHT,RSLV,RESOLVED,0.0,0.0,2114,1,1,P,_3F65,4,2,NaN
2,2015,EPD,LAW,15000003,2015-01-01 00:01:05.000,NaN,PHONE,CHECK WELFARE,ASST,ASSISTED,708.0,1094.0,1538,1,1,5,_3J79,1,1,NaN


In [64]:
eug_clean = eug[["calltime", "closed_as", "agency", "primeunit"]]
eug_clean = eug_clean.dropna()

#closed_as column
#change to CAHOOTS DEPLOYED: 'K897 DEPLOYED', 'K891 DEPLOYED', 'K895 DEPLOYED', 'K898 DEPLOYED', 'K896 DEPLOYED', 'K893 DEPLOYED'
eug_clean.replace(['K897 DEPLOYED', 'K891 DEPLOYED', 'K895 DEPLOYED', 'K898 DEPLOYED', 'K896 DEPLOYED', 'K893 DEPLOYED'], "CAHOOTS DEPLOYED", inplace=True)
#assumption this is hospital transport: 'PATIENT TRANSPORTED' and changed to 'HOSPITAL'
eug_clean.replace("PATIENT TRANSPORTED", "HOSPITAL", inplace=True)
#should 'JUVENILE TAKEN INTO CUSTODY' count as 'ARREST'?
eug_clean.replace("JUVENILE TAKEN INTO CUSTODY", "ARREST", inplace=True)


#change calltime to datetime objects
eug_clean["calltime"] = pd.to_datetime(eug_clean["calltime"], format="%Y-%m-%d %H:%M:%S.%f")


#Make is_CAHOOTS column
#True if agency==CAHE and/or primeunit follows the pattern _#J##
eug_clean["is_cahoots"] = (eug_clean["agency"] == "CAHE" ) | (eug_clean["primeunit"].str.match(r"^.\dJ\d{2}$", na=False))
eug_clean.head()

,calltime,closed_as,agency,primeunit,is_cahoots
0,2015-01-01 00:00:00,ASSISTED,EPD,_5E48,False
1,2015-01-01 00:00:44,RESOLVED,EPD,_3F65,False
2,2015-01-01 00:01:05,ASSISTED,EPD,_3J79,False
3,2015-01-01 00:03:16,PATROL CHECK,EPD,_5E48,False
4,2015-01-01 00:03:34,ADVISED,EPD,_5K97,False


# Springfield

In [ ]:
spd = pd.read_csv("2015-2025 SPD Calls for Service.csv")
spd_units = pd.read_csv("2015-2025 SPD Responding Units.csv")
spd_close = pd.read_csv("2015-2025 SPD Calls with Close Codes.csv")
designator = pd.read_csv("SPD Designator Notes.csv")

spd_new = spd.join(spd_close) #join on index

spd_clean = spd_new[["Incident Number", "Responding Agency", "Primary Responding Unit", "Call Creation Time", "Definition", "Call Year"]]
spd_clean = spd_clean.rename(columns={"Definition": "closed_as", "Incident Number": "inci id"})
spd_clean["closed_as"] = spd_clean["closed_as"].str.upper()

#merge on incident id
spd_clean1 = spd_clean.merge(spd_units, left_on="inci id", right_on="inci id", how="left")
spd_clean1.drop("Unnamed: 1", axis=1, inplace=True)
#convert to datetime
spd_clean1["calltime"] = pd.to_datetime(spd_clean1["Call time"], format="%m/%d/%y %H:%M")

,inci id,Responding Agency,Primary Responding Unit,Call Creation Time,closed_as,Call Year,call time,prime unit,units dispatched in order
0,15000074,SPD,1S18,16:21.0,EXECUTIVE ORDER 2012 VIOLATION,2015,01/01/2015 01:16,1S18,"1S18 ,1S21 ,K94"
1,15000083,SPD,3D2,22:29.0,DISREGARD,2015,01/01/2015 01:22,3D2,3D2
2,15000167,SPD,,15:28.0,ABANDONED VEHICLE,2015,01/01/2015 03:15,,NaN
3,15000216,SPD,1S22,27:24.0,ACTION TAKEN,2015,01/01/2015 05:27,1S22,1S22
4,15000222,SPD,1S11,52:53.0,ACCIDENTALLY CHOSE NEW EVENT,2015,01/01/2015 05:52,1S11,"1S11 ,1S12"
